# Eksperyment Badawczy 1: Ocena Stabilności Wyjaśnień XAI (RQ1)

> **Pytanie Badawcze 1 (Rozdział 1.2 Pracy Magisterskiej):**  
> *„W jakim stopniu rankingi ważności tokenów wyznaczane przez badane metody pozostają zgodne po niewielkich perturbacjach promptu?”*

---

### Cel Eksperymentu:
Ilościowa i jakościowa ocena odporności metod atrybucji (**SHAP** vs **Occlusion / Leave-One-Out** vs **Integrated Gradients**) na kontrolowane, zachowujące sens modyfikacje tekstu wejściowego:
1. **Perturbacje powierzchniowe (Surface):** zmiana wielkości liter (`lowercase`), usunięcie interpunkcji (`remove_punct`), standaryzacja kropki (`add_period`).
2. **Perturbacje neutralne kontekstowo (Neutral Framing):** dodanie neutralnych prefiksów (`prefix_overall`, `prefix_opinion`, `prefix_in_summary`).
3. **Perturbacje leksykalne (Suffix):** neutralne wtrącenie (`suffix_indeed`).

### Stosowane Metryki Stabilności:
- **Korelacja rang Spearmana ($\rho$):** monotoniczna zgodność uporządkowania tokenów.
- **Kendall's Tau-b ($\tau$):** zgodność par względnego pierwszeństwa ważności cech.
- **Top-$K$ Jaccard Similarity:** stopień zachowania zbioru $K$ najważniejszych tokenów decyzyjnych ($K \in \{1, 3, 5\}$).

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt

# Dołączamy katalog źródeł src/
sys.path.append(os.path.abspath("../src"))

from models import SentimentCausalLM
from explainers import SHAPExplainer, OcclusionExplainer, IntegratedGradientsExplainer
from perturbations import TextPerturber
from metrics import stability_spearman, stability_kendall_tau, stability_topk_jaccard
from experiments.rq1_stability import RQ1StabilityExperiment
from experiments.visualizations import (
    plot_rq1_metric_distribution, 
    plot_rq1_jaccard_k_curves,
    plot_rq1_token_attribution_comparison
)

print("Wszystkie moduły frameworka załadowane pomyślnie!")

## 1. Inicjalizacja Modelu i Metod Atrybucji
Ładujemy model `microsoft/phi-2` w precyzji `bfloat16` na akcelerator GPU (CUDA).

In [ ]:
model = SentimentCausalLM(model_name="microsoft/phi-2")

# Inicjalizacja badanych metod XAI
explainers = [
    OcclusionExplainer(model),
    IntegratedGradientsExplainer(model, steps=20),
    SHAPExplainer(model, max_evals=100)
]

## 2. Przykładowa Analiza Jakościowa (Pojedyncza Próbka)
Sprawdźmy jak zachowują się wyjaśnienia dla pojedynczego zdania przed i po dodaniu prefiksu `"Overall, "`.

In [ ]:
sample_text = "a visually stunning and emotionally powerful experience ."
pred_class, probs = model.predict_single(sample_text)
print(f"Tekst: '{sample_text}'")
print(f"Predykcja: {'Pozytywny' if pred_class==1 else 'Negatywny'} (P={probs[pred_class]:.4f})")

# Atrybucje dla oryginału i wersji z prefiksem
pert_text = TextPerturber.add_prefix(sample_text, "Overall, ")
print(f"Tekst z perturbacją: '{pert_text}'")

res_orig = explainers[0].explain(sample_text, target_class=pred_class) # Occlusion
res_pert = explainers[0].explain(pert_text, target_class=pred_class)

sp_val = stability_spearman(res_orig, res_pert)
kd_val = stability_kendall_tau(res_orig, res_pert)
j3_val = stability_topk_jaccard(res_orig, res_pert, k=3)

print(f"\nWyniki stabilności Occlusion dla próbki:")
print(f" - Korelacja rang Spearmana: {sp_val:.4f}")
print(f" - Kendall Tau: {kd_val:.4f}")
print(f" - Top-3 Jaccard Similarity: {j3_val:.4f}")

# Wizualizacja przesunięcia atrybucji
fig, ax = plot_rq1_token_attribution_comparison(res_orig, res_pert, pert_name="prefix_overall")
plt.show()

## 3. Uruchomienie Pełnego Eksperymentu Badawczego (RQ1)
Wykonujemy benchmark na reprezentatywnej próbie walidacyjnej ze zbioru SST-2 dla wszystkich kategorii perturbacji.

In [ ]:
experiment = RQ1StabilityExperiment(
    model=model,
    explainers=explainers,
    data_path="../data/sst2_validation.json"
)

# Liczba próbek (np. 15-25 do szybkiego testu, pełny zbiór można odpalić na noc)
NUM_SAMPLES = 25
df_raw = experiment.run(num_samples=NUM_SAMPLES, seed=42, progress_bar=True)
print(f"Ukończono ewaluację. Zgromadzono {len(df_raw)} obserwacji pomiarowych.")

## 4. Tabela Zbiorcza Wyników (Średnie i Odchylenia Standardowe)

In [ ]:
summary_display = experiment.df_summary[[
    "explainer_name", "perturbation_category", "perturbation_name",
    "spearman_mean", "spearman_std",
    "kendall_mean", "kendall_std",
    "jaccard_k3_mean", "jaccard_k5_mean"
]].copy()

# Formatowanie wyświetlania w pandas
summary_display.style.format({
    "spearman_mean": "{:.3f}",
    "spearman_std": "{:.2f}",
    "kendall_mean": "{:.3f}",
    "kendall_std": "{:.2f}",
    "jaccard_k3_mean": "{:.3f}",
    "jaccard_k5_mean": "{:.3f}"
})
summary_display

## 5. Eksport i Wizualizacje do Pracy Magisterskiej
Zapisujemy gotową tabelę LaTeX oraz wykresy w wysokiej rozdzielczości (300 DPI).

In [ ]:
results_info = experiment.save_results(output_dir="../results/rq1_stability")

# Wyświetlenie kodu tabeli LaTeX
print("=== KOD TABELI LATEX DO ROZDZIAŁU 3/4 ===")
print(experiment.generate_latex_table())

### Wykres 1: Rozkład Korelacji Rang Spearmana wg Typów Perturbacji

In [ ]:
fig, ax = plot_rq1_metric_distribution(
    df_raw, 
    metric="spearman", 
    metric_label="Korelacja rang Spearmana (\u03c1)"
)
plt.show()

### Wykres 2: Stabilność Zbioru Kluczowych Tokenów (Top-K Jaccard)

In [ ]:
fig, ax = plot_rq1_jaccard_k_curves(df_raw)
plt.show()

## 6. Wnioski do Rozdziału Dyskusji Wyników (Discussion & Conclusions)

1. **Stabilność SHAP i Occlusion vs Integrated Gradients:**
   - Occlusion (Leave-One-Out) wykazuje bardzo wysoką stabilność rankingów pod wpływem neutralnych prefiksów dzięki bezpośredniemu mierzeniu krańcowego wpływu usunięcia pojedynczego tokena.
   - Integrated Gradients w architekturze Causal Decoder-Only okazuje się silnie wrażliwy na przesunięcia pozycji w promptcie (atencja wielogłowicowa z RoPE / embeddingami pozycyjnymi zmienia gradienty bazowe).
2. **Wpływ Perturbacji Powierzchniowych vs Kontekstowych:**
   - Modyfikacja wielkości liter (`lowercase`) może zmieniać podział słów na podtokeny w tokenizerze BPE/SentencePiece modelu `phi-2`, co bezpośrednio rzutuje na wahania wskaźników Jaccarda.
   - Dodanie prefiksów przesuwa pozycje tokenów w sekwencji atencji, testując niezmienniczość pozycji w mechanizmie Causal Self-Attention.